In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, T5EncoderModel

tokenizer = AutoTokenizer.from_pretrained("ai-forever/FRIDA")
model = T5EncoderModel.from_pretrained("ai-forever/FRIDA")



/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 219/219 [00:00<00:00, 17945.39it/s]


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)


In [3]:
def predict(texts, batch_size=4):
    texts = [f"categorize: {t}" for t in texts]
    text_emb = encode_batch(texts, batch_size=batch_size)
    scores = text_emb @ label_emb.T
    probs = torch.sigmoid(scores)
    return probs

In [4]:
import pandas as pd

In [5]:
df=pd.read_csv("new_ds.csv")
df

,text,ASSORTMENT,PROMOTIONS,DELIVERY,PRICE,PRODUCTS_QUALITY,SUPPORT,CATALOG_NAVIGATION,PAYMENT
0,"Маленький выбор товаров, хотелось бы ассортиме...",1,0,0,0,0,0,0,0
1,Быстро,0,0,1,0,0,0,0,0
2,Доставка постоянно задерживается,0,0,1,0,0,0,0,0
3,Наценка и ассортимент расстраивают,1,0,0,1,0,0,0,0
4,Можно немного скинуть минимальную сумму заказа...,0,0,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...
2277,"Очень расстраивает, что сервис стал постоянно ...",0,0,1,0,0,0,0,0
2278,Хлеб привезли не свежий,0,0,0,0,1,0,0,0
2279,"Сервис испортился,доставка по [NUM],[NUM] часа(",0,0,1,0,0,0,0,0
2280,Последнее время постоянные проблемы с доставко...,0,0,1,0,0,0,0,0


In [6]:
label_texts = [
    "category: проблемы с доставкой, долгая доставка, курьер",
    "category: акции, скидки, промокоды",
    "category: ассортимент товаров, выбор",
    "category: высокая или низкая цена",
    "category: качество товара, брак",
    "category: служба поддержки, помощь клиенту",
    "category: навигация по каталогу, поиск товаров",
    "category: оплата, способы оплаты, проблемы с оплатой"
]

In [ ]:
def pool(hidden_state, mask):
    s = torch.sum(hidden_state * mask.unsqueeze(-1).float(), dim=1)
    d = mask.sum(axis=1, keepdim=True).float()
    return s / d

def encode_batch(texts, batch_size=8):
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tokens = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**tokens)
        emb = pool(outputs.last_hidden_state, tokens["attention_mask"])
        embs.append(F.normalize(emb, p=2, dim=1))
    return torch.cat(embs, dim=0).cpu().numpy()  # на CPU для sklearn

In [13]:
text=df["text"]
y_true=df.drop(columns=["text"])

In [14]:
from sklearn.model_selection import train_test_split


In [ ]:
X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    text, y_true, test_size=0.2, random_state=42
)

X_train_texts = X_train_texts.astype(str).tolist()
X_val_texts = X_val_texts.astype(str).tolist()

X_train = encode_batch(X_train_texts)
X_val = encode_batch(X_val_texts)


In [20]:
X_train_np = X_train.cpu().numpy() if X_train.is_cuda else X_train.numpy()
X_val_np = X_val.cpu().numpy() if X_val.is_cuda else X_val.numpy()

In [22]:
import lightgbm as lgb
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score



base_clf = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    class_weight='balanced',  # сбалансирует 0 и 1 автоматически
    random_state=42
)

clf = MultiOutputClassifier(base_clf)
# обучаем
clf.fit(X_train_np, y_train)

# предсказания
y_pred = clf.predict(X_val_np)


[LightGBM] [Info] Number of positive: 216, number of negative: 1609
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020255 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 387344
[LightGBM] [Info] Number of data points in the train set: 1825, number of used features: 1519
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [39]:
y_val

,ASSORTMENT,PROMOTIONS,DELIVERY,PRICE,PRODUCTS_QUALITY,SUPPORT,CATALOG_NAVIGATION,PAYMENT
1268,0,0,0,0,0,0,1,0
1633,0,0,1,0,0,1,0,0
700,0,0,1,0,1,1,0,0
2028,0,0,1,0,0,0,0,0
596,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...
387,0,0,1,0,0,0,0,0
649,0,0,0,0,1,0,0,0
1645,0,0,0,0,0,0,0,0
1318,0,0,1,0,1,0,1,0


In [24]:
import numpy as np

y_probs = np.array([est.predict_proba(X_val_np)[:, 1] for est in clf.estimators_]).T


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [102]:
y_pred_proba = clf.predict_proba(X_val_np)  # список массивов для каждой метки
y_pred_thresh = np.zeros_like(y_val)

thresholds = [0.2, 0.01781436, 0.28073387, 0.04927171, 0.14049791, 0.02565653, 0.00265701, 0.00237321]  # подбираются эмпирически

for i, th in enumerate(thresholds):
    y_pred_thresh[:, i] = (y_pred_proba[i][:, 1] >= th).astype(int)

f1 = f1_score(y_val, y_pred_thresh, average='macro')
print(f"Macro F1 с порогами: {f1:.3f}")

Macro F1 с порогами: 0.741


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning

In [ ]:
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score
import numpy as np



def macro_f1(thresholds):
    y_pred = np.zeros_like(y_val_np)
    for i, th in enumerate(thresholds):
        y_pred[:, i] = (y_pred_proba[i][:, 1] >= th).astype(int)
    return -f1_score(y_val_np, y_pred, average='macro') 

bounds = [(0, 1)] * len(y_pred_proba)
result = differential_evolution(macro_f1, bounds)
optimal_thresholds = result.x

# Применяем
y_pred_thresh = np.zeros_like(y_val_np)
for i, th in enumerate(optimal_thresholds):
    y_pred_thresh[:, i] = (y_pred_proba[i][:, 1] >= th).astype(int)

f1 = f1_score(y_val_np, y_pred_thresh, average='macro')
print("Максимальный Macro F1:", f1)
print("Порог для каждой метки:", optimal_thresholds)

Максимальный Macro F1: 0.740758915204819
Порог для каждой метки: [0.24442053 0.01781436 0.28073387 0.04927171 0.14049791 0.02565653
 0.00265701 0.00237321]


In [103]:
from sklearn.metrics import classification_report
print(classification_report(y_val,y_pred_thresh))

              precision    recall  f1-score   support

           0       0.91      0.76      0.83        41
           1       1.00      0.57      0.73        14
           2       0.91      0.95      0.93       243
           3       0.86      0.89      0.88        84
           4       0.93      0.90      0.92        93
           5       0.77      0.72      0.75        57
           6       0.42      0.38      0.40        34
           7       1.00      0.33      0.50         9

   micro avg       0.87      0.85      0.86       575
   macro avg       0.85      0.69      0.74       575
weighted avg       0.87      0.85      0.85       575
 samples avg       0.74      0.73      0.72       575



/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, m

In [107]:
from sklearn.metrics import hamming_loss

hloss = hamming_loss(y_val_np, y_pred_thresh)
print("Hamming Loss:", hloss)

Hamming Loss: 0.04431072210065645


In [89]:
from sklearn.metrics import classification_report
print(classification_report(y_val,y_pred_thresh))

              precision    recall  f1-score   support

           0       0.91      0.76      0.83        41
           1       1.00      0.57      0.73        14
           2       0.91      0.95      0.93       243
           3       0.86      0.89      0.88        84
           4       0.93      0.90      0.92        93
           5       0.77      0.72      0.75        57
           6       0.42      0.38      0.40        34
           7       1.00      0.33      0.50         9

   micro avg       0.87      0.85      0.86       575
   macro avg       0.85      0.69      0.74       575
weighted avg       0.87      0.85      0.85       575
 samples avg       0.74      0.73      0.72       575



/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, m

In [106]:
new_text_features = encode_batch(["Слишком дорого"]).cpu().numpy()

new_probs = clf.predict_proba(new_text_features)

new_pred = np.zeros((new_text_features.shape[0], len(thresholds)), dtype=int)
for i, th in enumerate(thresholds):
    new_pred[:, i] = (new_probs[i][:, 1] >= th).astype(int)

print(new_pred)

[[0 0 0 1 0 0 0 0]]


/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/sasha/Python/VKR/pytorch_bert/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning